# 02 — Train DAN baseline

Prereqs: run `01_colab_setup.ipynb` first so deps are installed and data is unpacked.

This notebook:
1. Clones `yaoing/DAN` into `third_party/DAN/` (gitignored).
2. Downloads the released RAF-DB checkpoint into `checkpoints/dan_rafdb.pth`.
3. Runs `python -m src.train --config configs/dan_rafdb.yaml`.
4. Evaluates the best checkpoint on the RAF-DB test set.

Target: WAR ≥ 87% on the official RAF-DB test split (3,068 images).
Report quotes 89.70% (report.md:271) — a 1–2 pt gap is acceptable on free-tier T4 with finetune-only.

In [ ]:
%cd /content/fer
import os
os.makedirs('third_party', exist_ok=True)
if not os.path.exists('third_party/DAN'):
    !git clone https://github.com/yaoing/DAN.git third_party/DAN
!ls third_party/DAN

In [ ]:
# Download released DAN checkpoint. The repo's README has a Google Drive link;
# typical filename is `affecnet8_epoch5_acc0.6209.pth` or `rafdb_epoch21_acc0.897_bacc0.8275.pth`.
# Download manually (Drive permission), upload to MyDrive/fer-data/checkpoints/, and copy here.
import shutil
from pathlib import Path
Path('checkpoints').mkdir(exist_ok=True)
src = Path('/content/drive/MyDrive/fer-data/checkpoints/dan_rafdb.pth')
dst = Path('checkpoints/dan_rafdb.pth')
if src.exists() and not dst.exists():
    shutil.copy(src, dst)
    print('Copied DAN checkpoint:', dst)
elif dst.exists():
    print('DAN checkpoint already present:', dst)
else:
    print(f'WARNING: no DAN checkpoint at {src}. Edit configs/dan_rafdb.yaml to set pretrained_ckpt: null to train from MS-Celeb-1M init.')

In [ ]:
!python -m src.train --config configs/dan_rafdb.yaml

In [ ]:
!python -m src.eval --config configs/dan_rafdb.yaml --ckpt runs/dan_rafdb/best.pth

In [ ]:
from IPython.display import Image
Image('runs/dan_rafdb/eval/confusion_matrix.png')